# Module 13 - Decorators Drills

---

### Instructions
1. Read each exercise carefully
2. Write your solution in the blank code cell below it
3. Run the **Testing** section when done
4. All tests green = ready for the next module

---

## Part 1 — Basic Decorators (Exercises 1-3)


### Exercise 1

Write a decorator `@uppercase_result` that:
- Calls the decorated function
- If the result is a string, returns it in UPPER CASE
- If the result is anything else, returns it unchanged
- Uses `@wraps`

Apply it to `get_severity_label(cvss_score)` that returns:
`'critical'` for >= 9.0, `'high'` for >= 7.0, `'medium'` for >= 4.0, `'low'` otherwise.

Assign:
- `label_critical` = `get_severity_label(9.8)` — should be `'CRITICAL'`
- `label_high` = `get_severity_label(7.5)` — should be `'HIGH'`
- `fn_name` = `get_severity_label.__name__` — should be `'get_severity_label'`


In [ ]:
# Exercise 1
from functools import wraps


### Exercise 2

Write a decorator `@count_calls` that:
- Tracks total number of times the decorated function is called
- After each call, prints: `[COUNT] {function_name} has been called {n} time(s)`
- Uses `@wraps` and `nonlocal`

Apply to `check_ip_reputation(ip_address)` that returns `True` if the IP
starts with `'185.'` or `'45.'` (known attacker ranges).

Call it 4 times with different IPs.
Assign `reputation_result` = result of calling it with `'185.220.101.47'`.


In [ ]:
# Exercise 2
from functools import wraps


### Exercise 3

Write a decorator `@block_on_none` that:
- Calls the decorated function
- If the result is `None`, prints `[BLOCKED] {function_name} returned None — execution halted`
  and raises `ValueError(f'{function_name} must not return None')`
- Otherwise returns the result normally
- Uses `@wraps`

Apply to `get_alert_handler(alert_type)` that returns a string like `'EMAIL'` for
`'CRITICAL'`, `'SLACK'` for `'HIGH'`, and `None` for unknown types.

Test with `'CRITICAL'`, `'HIGH'`, `'UNKNOWN'`.
Assign `handler_critical` = result for `'CRITICAL'` (should be `'EMAIL'`).


In [ ]:
# Exercise 3
from functools import wraps


---

## Part 2 — `*args`, `**kwargs`, and `@wraps` (Exercises 4-6)


### Exercise 4

Write a decorator `@log_call` that:
- Before calling the function: prints `[LOG] >> {function_name}(args={args}, kwargs={kwargs})`
- After: prints `[LOG] << {function_name} returned {result}`
- Works for any function signature (use `*args, **kwargs`)
- Uses `@wraps`

Apply to:
- `scan_host(hostname, port_range=(1, 1024))` — returns `f'{hostname}:{port_range}'`
- `classify_event(event_type, source_ip, severity='MEDIUM')` — returns a dict
  `{"event": event_type, "ip": source_ip, "severity": severity}`

Call each once. Assign `scan_result` and `event_result` to the return values.


In [ ]:
# Exercise 4
from functools import wraps


### Exercise 5

Write a decorator `@validate_cvss` that:
- Checks that the first positional argument is a float or int between 0.0 and 10.0
- If valid: calls the function normally
- If invalid: raises `ValueError(f'CVSS score {value} must be between 0.0 and 10.0')`
- Uses `@wraps` and `*args, **kwargs`

Apply to `generate_cve_report(cvss_score, cve_id)` that returns
`{"cve": cve_id, "cvss": cvss_score, "severity": "CRITICAL" if cvss_score >= 9.0 else "OTHER"}`.

Assign:
- `report_valid` = result for `(9.8, 'CVE-2021-44228')` — no error
- `raised_for_invalid` = `True` if calling with `(11.0, 'CVE-BAD')` raises `ValueError`


In [ ]:
# Exercise 5
from functools import wraps


### Exercise 6

Confirm `@wraps` is working correctly for the following scenario:

Write `@enforce_https` — a decorator that:
- Checks that the first argument (a URL string) starts with `'https://'`
- If not: raises `ValueError('Only HTTPS connections are permitted')`
- Uses `@wraps`

Apply to `fetch_threat_feed(url, timeout=10)` with docstring:
`'Download threat intelligence feed from a URL.'`

Assign:
- `fn_name` = `fetch_threat_feed.__name__` — must be `'fetch_threat_feed'`
- `fn_doc` = `fetch_threat_feed.__doc__` — must equal the docstring above
- `raised_for_http` = `True` if calling with `'http://feed.example.com'` raises `ValueError`


In [ ]:
# Exercise 6
from functools import wraps


---

## Part 3 — Parameterized Decorators (Exercises 7-9)


### Exercise 7

Write `@max_calls(n)` — a parameterized decorator that:
- Allows the decorated function to be called at most `n` times
- On the `n+1`th call: raises `RuntimeError(f'{function_name} call limit of {n} exceeded')`
- Uses `nonlocal`, `@wraps`

Apply `@max_calls(3)` to `open_soc_ticket(incident_id)` that just prints
`Ticket opened for: {incident_id}`.

Assign:
- `raised_on_4th` = `True` if the 4th call raises `RuntimeError`


In [ ]:
# Exercise 7
from functools import wraps


### Exercise 8

Write `@require_min_cvss(threshold)` — a parameterized decorator that:
- Checks the first argument (a float CVSS score)
- If score >= threshold: calls the function and returns the result
- If score < threshold: returns the string `f'SKIPPED: CVSS {score} below threshold {threshold}'`
  without calling the function
- Uses `@wraps`

Apply:
- `@require_min_cvss(9.0)` on `alert_critical_team(cvss_score, cve_id)` → returns `f'Alerted for {cve_id}'`
- `@require_min_cvss(7.0)` on `create_patch_ticket(cvss_score, cve_id)` → returns `f'Ticket: {cve_id}'`

Assign:
- `critical_alerted` = `alert_critical_team(9.8, 'CVE-2021-44228')` — should be `'Alerted for CVE-2021-44228'`
- `critical_skipped` = `alert_critical_team(7.5, 'CVE-2023-44487')` — should be the SKIPPED string
- `patch_created` = `create_patch_ticket(7.5, 'CVE-2023-44487')` — should be `'Ticket: CVE-2023-44487'`


In [ ]:
# Exercise 8
from functools import wraps


### Exercise 9

Write `@tag_result(tag)` — a parameterized decorator that:
- Calls the decorated function
- If the result is a dict, adds a key `'_tag'` with value `tag` before returning
- If the result is not a dict, wraps it: returns `{"result": original_result, "_tag": tag}`
- Uses `@wraps`

Apply:
- `@tag_result('threat-intel')` to `lookup_ip(ip)` that returns `{"ip": ip, "score": 90}`
- `@tag_result('port-scan')` to `scan_port(port)` that returns `True` if port in `{22, 80, 443}`

Assign:
- `ip_result` = `lookup_ip('185.220.101.47')` — must be a dict with `'_tag': 'threat-intel'`
- `port_result` = `scan_port(443)` — must be a dict with `'_tag': 'port-scan'`


In [ ]:
# Exercise 9
from functools import wraps


---

## Part 4 — Stacking Decorators (Exercises 10-12)


### Exercise 10

Write two decorators:
- `@measure_time` — prints `[TIME] {function_name}: {ms}ms` after the call
- `@log_exceptions` — catches any exception, prints `[EXCEPTION] {name}: {type}: {msg}`, returns `None`

Stack them (think about order: which should be outermost?) on:
`risky_operation(value)` — raises `ZeroDivisionError` if `value == 0`, else returns `100 / value`.

Assign:
- `safe_result` = `risky_operation(4)` — should be `25.0`
- `error_result` = `risky_operation(0)` — should be `None` (exception caught)


In [ ]:
# Exercise 10
from functools import wraps
import time


### Exercise 11

Stack three decorators on `process_alert(alert_id, severity, source_ip)`:
- `@audit_log` — logs `[AUDIT] {name} called by {source_ip}`
  (hint: `source_ip` is the 3rd positional argument or keyword arg)
- `@validate_ip` (first positional arg must be a valid IP — or use `source_ip` keyword)
- `@require_min_severity(7.0)` — first argument here is `severity` (as a float)

Wait — there's a design challenge: the arguments don't match what each decorator expects.
Think about what order and approach makes the most sense, then implement it.

The function should print: `Processing alert {alert_id} from {source_ip} [{severity}]`

Test with valid and invalid inputs. Assign `process_result` = result of a valid call.


In [ ]:
# Exercise 11
from functools import wraps


### Exercise 12

Stack `@cache_result` and `@measure_time` on `expensive_lookup(cve_id)` that
simulates an expensive database call (use `time.sleep(0.05)`) and returns
`{"cve": cve_id, "cvss": 9.8}`.

Call it 3 times with the same `cve_id`, then once with a different one.

Assign:
- `first_call_result` = first call result
- `second_call_result` = second call result (should be identical to first)
- `same_result` = `True` if `first_call_result == second_call_result`

The timing output should show the 2nd and 3rd calls are much faster than the 1st.


In [ ]:
# Exercise 12
from functools import wraps
import time


---

## Part 5 — Putting It Together (Exercises 13-15)


### Exercise 13

Build a `@sanitize_input` decorator that:
- Checks all string arguments (both positional and keyword)
- Raises `ValueError('Potential injection detected')` if any string contains
  any of these characters: `';'`, `'--'`, `'<script'`, `'DROP '` (case-insensitive)
- Otherwise calls the function normally
- Uses `@wraps`

Apply to `run_query(query, database='main')` that just returns `f'Running: {query} on {database}'`.

Assign:
- `safe_query` = result of a clean call
- `injection_blocked` = `True` if `run_query("SELECT * FROM users; DROP TABLE users")` raises `ValueError`


In [ ]:
# Exercise 13
from functools import wraps


### Exercise 14

Write a complete decorator `@security_gate(role, min_cvss)` that combines:
1. Role check: the function must be called with `role=` keyword arg matching `role`
   — if not, print `[DENIED] Insufficient role` and return `None`
2. CVSS check: first positional arg must be >= `min_cvss`
   — if not, print `[DENIED] CVSS below threshold` and return `None`
3. Audit log: on success, print `[GATE PASSED] {function_name} called by role={role}`

Apply `@security_gate(role='admin', min_cvss=9.0)` to
`deploy_emergency_patch(cvss_score, cve_id, role)` which returns `f'Deployed: {cve_id}'`.

Assign:
- `patch_deployed` = result with `(9.8, 'CVE-2021-44228', role='admin')` — should be `'Deployed: CVE-2021-44228'`
- `denied_role` = result with `(9.8, 'CVE-2021-44228', role='intern')` — should be `None`
- `denied_cvss` = result with `(7.5, 'CVE-2023-44487', role='admin')` — should be `None`


In [ ]:
# Exercise 14
from functools import wraps


### Exercise 15 — Full Decorator Pipeline

Build a production-ready function `assess_and_respond(cvss_score, cve_id, analyst_role)`
decorated with a complete security pipeline:

1. `@measure_time` — outer: always know how long each assessment takes
2. `@log_exceptions` — catch and log any unexpected errors
3. `@rate_limit(max_calls=10)` — prevent runaway automation
4. `@require_min_cvss(7.0)` — skip low-severity CVEs automatically

The function body should:
- Classify severity (CRITICAL >= 9.0, HIGH >= 7.0)
- Determine response action: CRITICAL → `'EMERGENCY_PATCH'`, HIGH → `'SCHEDULE_PATCH'`
- Return a dict: `{"cve": cve_id, "cvss": cvss_score, "severity": ..., "action": ..., "analyst": analyst_role}`

Call it with:
- `(9.8, 'CVE-2021-44228', 'senior_analyst')` — should return a full dict
- `(3.1, 'CVE-2023-00001', 'junior_analyst')` — should be skipped (below threshold)

Assign:
- `critical_response` = result for the 9.8 call
- `low_skipped` = result for the 3.1 call — should be the SKIPPED string
- `fn_name_preserved` = `assess_and_respond.__name__` — must be `'assess_and_respond'`


In [ ]:
# Exercise 15
from functools import wraps
import time


---

## Testing

Run the cell below. **Don't worry about the `class Test...` syntax** — `unittest` is covered later.


In [ ]:
import unittest

class TestDecoratorDrills(unittest.TestCase):

    # ---- Part 1 ----

    def test_01_uppercase_result(self):
        self.assertEqual(label_critical, "CRITICAL",
            "label_critical must be 'CRITICAL' (uppercase) — @uppercase_result should upper() strings")
        self.assertEqual(label_high, "HIGH",
            "label_high must be 'HIGH'")
        self.assertEqual(fn_name, "get_severity_label",
            "fn_name must be 'get_severity_label' — @wraps must be applied")

    def test_02_count_calls(self):
        self.assertIsInstance(reputation_result, bool,
            "reputation_result must be a bool — check_ip_reputation returns True/False")
        self.assertTrue(reputation_result,
            "check_ip_reputation('185.220.101.47') must return True — starts with '185.'")

    def test_03_block_on_none(self):
        self.assertEqual(handler_critical, "EMAIL",
            "handler_critical must be 'EMAIL' for CRITICAL severity")
        raised = False
        try:
            get_alert_handler("UNKNOWN")
        except ValueError:
            raised = True
        self.assertTrue(raised,
            "@block_on_none must raise ValueError when the function returns None")

    # ---- Part 2 ----

    def test_04_log_call(self):
        self.assertIsInstance(scan_result, str,
            "scan_result must be a string")
        self.assertIsInstance(event_result, dict,
            "event_result must be a dict")
        self.assertIn("event", event_result,
            "event_result dict must have key 'event'")

    def test_05_validate_cvss(self):
        self.assertIsInstance(report_valid, dict,
            "report_valid must be a dict")
        self.assertEqual(report_valid["cvss"], 9.8,
            "report_valid['cvss'] must be 9.8")
        self.assertTrue(raised_for_invalid,
            "raised_for_invalid must be True — @validate_cvss must raise ValueError for CVSS > 10")

    def test_06_wraps_metadata(self):
        self.assertEqual(fn_name, "fetch_threat_feed",
            "fn_name must be 'fetch_threat_feed' — @wraps must preserve __name__")
        self.assertIsNotNone(fn_doc,
            "fn_doc must not be None — @wraps must preserve __doc__")
        self.assertIn("threat feed", fn_doc.lower(),
            "fn_doc must contain the original docstring text")
        self.assertTrue(raised_for_http,
            "raised_for_http must be True — @enforce_https must raise ValueError for http:// URLs")

    # ---- Part 3 ----

    def test_07_max_calls(self):
        self.assertTrue(raised_on_4th,
            "raised_on_4th must be True — @max_calls(3) must raise RuntimeError on 4th call")

    def test_08_require_min_cvss(self):
        self.assertEqual(critical_alerted, "Alerted for CVE-2021-44228",
            "critical_alerted must be 'Alerted for CVE-2021-44228'")
        self.assertIsInstance(critical_skipped, str,
            "critical_skipped must be a string (the SKIPPED message)")
        self.assertIn("SKIPPED", critical_skipped,
            "critical_skipped must contain 'SKIPPED'")
        self.assertEqual(patch_created, "Ticket: CVE-2023-44487",
            "patch_created must be 'Ticket: CVE-2023-44487' — 7.5 >= 7.0 threshold")

    def test_09_tag_result(self):
        self.assertIsInstance(ip_result, dict,
            "ip_result must be a dict")
        self.assertEqual(ip_result.get("_tag"), "threat-intel",
            "ip_result must have '_tag': 'threat-intel'")
        self.assertIsInstance(port_result, dict,
            "port_result must be a dict")
        self.assertEqual(port_result.get("_tag"), "port-scan",
            "port_result must have '_tag': 'port-scan'")

    # ---- Part 4 ----

    def test_10_stacked_time_exception(self):
        self.assertEqual(safe_result, 25.0,
            "safe_result must be 25.0 — 100 / 4")
        self.assertIsNone(error_result,
            "error_result must be None — @log_exceptions catches ZeroDivisionError")

    def test_11_three_decorators(self):
        self.assertIsNotNone(process_result,
            "process_result must not be None — valid call should succeed")

    def test_12_cache_and_time(self):
        self.assertIsInstance(first_call_result, dict,
            "first_call_result must be a dict")
        self.assertTrue(same_result,
            "same_result must be True — cached result must equal original result")
        self.assertEqual(first_call_result, second_call_result,
            "first_call_result must equal second_call_result — same cache entry")

    # ---- Part 5 ----

    def test_13_sanitize_input(self):
        self.assertIsInstance(safe_query, str,
            "safe_query must be a string")
        self.assertTrue(injection_blocked,
            "injection_blocked must be True — SQL injection attempt must raise ValueError")

    def test_14_security_gate(self):
        self.assertEqual(patch_deployed, "Deployed: CVE-2021-44228",
            "patch_deployed must be 'Deployed: CVE-2021-44228' — admin + CVSS 9.8 passes gate")
        self.assertIsNone(denied_role,
            "denied_role must be None — 'intern' role should not pass the gate")
        self.assertIsNone(denied_cvss,
            "denied_cvss must be None — CVSS 7.5 below 9.0 threshold should be denied")

    def test_15_full_pipeline(self):
        self.assertIsInstance(critical_response, dict,
            "critical_response must be a dict")
        self.assertEqual(critical_response.get("action"), "EMERGENCY_PATCH",
            "action for CVSS 9.8 must be 'EMERGENCY_PATCH'")
        self.assertIsInstance(low_skipped, str,
            "low_skipped must be a string (the SKIPPED message for CVSS 3.1)")
        self.assertIn("SKIPPED", low_skipped,
            "low_skipped must contain 'SKIPPED'")
        self.assertEqual(fn_name_preserved, "assess_and_respond",
            "fn_name_preserved must be 'assess_and_respond' — @wraps must preserve __name__")


unittest.main(argv=[""], verbosity=2, exit=False)


---

## Well Done!

If all 15 tests pass, you can write reusable, composable decorator pipelines —
the same pattern used in Flask, FastAPI, and every major security framework to
enforce authentication, rate limiting, and audit logging at the function level.

**Next module:** **[Typing](../05_typing/00_Typing.ipynb)** — type hints, annotations, and `mypy` for building
documented, maintainable security tooling that other engineers can actually read.
